In [3]:
import io
import json
import boto3
import pandas as pd
import os
from botocore.config import Config
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Trims and normalizes all column headers (lowercase, stripped, spaces to underscores).
    2. Trims leading/trailing whitespace from string/text columns.
    """
    # Step 1: Clean column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    
    # Step 2: Clean text cell content for string columns
    for col in df.columns:
        if df[col].dtype == 'object' or isinstance(df[col].dtype, pd.StringDtype):
            # Check if elements are strings before applying strip
            df[col] = df[col].astype(str).str.strip()
            
    return df

In [6]:

    config = Config(read_timeout=600, connect_timeout=600)
    s3_client = boto3.client('s3', config=config)
    bucket_name = "shvnsh-rearc-quest"
    
    print("--- Phase 1: Processing BLS Time Series Data ---")
    
    # Read the tab-separated BLS file from S3 into memory
    bls_key = "data/bls_data/pr.data.0.Current"
    bls_obj = s3_client.get_object(Bucket=bucket_name, Key=bls_key)
    
    # read_csv handles the tab delimiter and schema inference automatically
    pr_df = pd.read_csv(io.BytesIO(bls_obj['Body'].read()), sep='\t')
    
    # Clean the dataframe columns and contents
    pr_cleaned_df = clean_dataframe(pr_df)
    
    # Log column schemas
    for col, dtype in zip(pr_cleaned_df.columns, pr_cleaned_df.dtypes):
        print(f"Column: {col}, Data Type: {dtype}")
        
    
    # Filter for the specific series ID and period
    filtered_ts = pr_cleaned_df[
        (pr_cleaned_df['series_id'] == "PRS30006032") & 
        (pr_cleaned_df['period'] == "Q01")
    ].copy()
    
    
    
    
    
    print("\n--- Phase 1: Processing DataUSA Population Data ---")
    
    # Read the population JSON file from S3
    pop_key = "data/datausa/population_data.json"
    pop_obj = s3_client.get_object(Bucket=bucket_name, Key=pop_key)
    raw_json = json.loads(pop_obj['Body'].read().decode('utf-8'))
    
    # Flatten the struct array wrapped in the 'data' key (mimicking Spark's explode)
    final_df = pd.DataFrame(raw_json['data'])
    
    # Normalize data types: Cast Population to Integer, ensure Year is numeric matching BLS
    final_df['Population'] = final_df['Population'].astype(int)
    final_df['Year'] = pd.to_numeric(final_df['Year'])
    
    # Run structural cleaning step
    population_df = clean_dataframe(final_df)
    
    # Filter population metrics between 2013 and 2018 inclusive
    filtered_population_df = population_df[population_df['year'].between(2013, 2018)]
    
    # Calculate aggregate summary stats (Mean and Standard Deviation)
    mean_pop = filtered_population_df['population'].mean()
    std_pop = filtered_population_df['population'].std()
    
    # Format metrics cleanly with commas and two decimal places
    print("\n[Population Stats (2013 - 2018)]:")
    print(f"Mean Population:   {mean_pop:,.2f}")
    print(f"StdDev Population: {std_pop:,.2f}")
    

    # Part 2 Report: Find the year with the maximum total value per series_id
    # 1. Group by series_id and year, summing the values
    yearly_sum_df = pr_cleaned_df.groupby(['series_id', 'year'], as_index=False)['value'].sum()
    
    # 2. Sort values to replicate window function ordering (value descending)
    yearly_sum_df = yearly_sum_df.sort_values(by=['series_id', 'value'], ascending=[True, False])
    
    # 3. Deduplicate by series_id keeping the first match (highest value due to sort) to get rank == 1
    report_df_1 = yearly_sum_df.drop_duplicates(subset=['series_id'], keep='first')
    
    print("\n[Highest Total Value Year Per Series]:")
    print(report_df_1.head(2000).to_string(index=False))



    
    print("\n--- Phase 3: Joining Datasets ---")
    
    # Ensure both dataframes share identical data types on the join key
    filtered_ts['year'] = filtered_ts['year'].astype(int)
    population_df['year'] = population_df['year'].astype(int)
    
    # Perform inner merge on matching 'year' fields
    joined_report_df = pd.merge(
        filtered_ts,
        population_df,
        on="year",
        how="inner"
    )
    
    print("\n[Final Merged Report Output]:")
    print(joined_report_df[['series_id','year','period', 'value', 'population']].to_string(index=False))
    
    # Option to push output back to S3 as an analytical CSV asset
    # csv_buffer = io.StringIO()
    # joined_report_df.to_csv(csv_buffer, index=False)
    # s3_client.put_object(Bucket=bucket_name, Key="data/reports/final_analysis_output.csv", Body=csv_buffer.getvalue())


--- Phase 1: Processing BLS Time Series Data ---
Column: series_id, Data Type: str
Column: year, Data Type: int64
Column: period, Data Type: str
Column: value, Data Type: float64
Column: footnote_codes, Data Type: str

--- Phase 1: Processing DataUSA Population Data ---

[Population Stats (2013 - 2018)]:
Mean Population:   322,069,808.00
StdDev Population: 4,158,441.04

[Highest Total Value Year Per Series]:
  series_id  year    value
PRS30006011  2022   20.500
PRS30006012  2022   17.100
PRS30006013  1998  705.895
PRS30006021  2010   17.700
PRS30006022  2010   12.400
PRS30006023  2014  503.216
PRS30006031  2022   20.600
PRS30006032  2021   17.000
PRS30006033  1998  702.672
PRS30006061  2022   34.500
PRS30006062  2021   29.800
PRS30006063  2025  665.334
PRS30006081  2021   24.500
PRS30006082  2021   24.500
PRS30006083  2022  131.294
PRS30006091  2002   43.400
PRS30006092  2002   44.300
PRS30006093  2013  514.158
PRS30006101  2020   33.100
PRS30006102  2020   35.700
PRS30006103  2025  67